# StrandsLingo - Your Agentic Translation Agent

## Overview
TODO

### Architecture

## Key Features


## Setup and Prerequisites

### Prerequisites
To execute this lab you will need:
* Python 3.11+
* AWS account with AWS credentials
* Anthropic Claude 4.5 enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Knowledge Base and Amazon S3 buckets

When running the lab in SageMaker AI Studio, in the upper right, select the Python 3 kernel (ipykernel). 

In [ ]:
# When running the lab in your own environment, create and acivate a Python vertual environment prior to running the next pip install cell.
# Uncomment the following two lines and run this cell if you are not using the SageMaker AI Studio
# python -m venv .venv
# source .venv/bin/activate

Install the required Python packages

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Introducing Customization

### Translation Memory Knowledge Base

##### Upload Data

In [31]:
import boto3
import json
import time
import uuid
from botocore.exceptions import ClientError

# Configuration
embedding_model = 'cohere.embed-multilingual-v3'
unique_id = str(uuid.uuid4())[:8]
region = "us-east-1"
profile_name = "demo-profile"

# Get account info
session = boto3.session.Session(
    profile_name=profile_name,
    region_name=region
)
account_id = session.client("sts").get_caller_identity()["Account"]

print(f"Account: {account_id}")
print(f"Region: {region}")
print(f"ID: {unique_id}")

Account: 986528949439
Region: us-east-1
ID: 2b879629


In [ ]:
# Resource names
bucket_name = f's3-translation-memory-{account_id}-{region}-{unique_id}'
vector_bucket_name = f's3-vectors-embeddings-{unique_id}'
index_name = f"s3-vectors-index-{unique_id}"
kb_role_name = f's3-vectors-kb-role-{unique_id}'
kb_name = f's3-vectors-kb-{unique_id}'

print(f"S3 Bucket: {bucket_name}")
print(f"Vector Bucket: {vector_bucket_name}")
print(f"Vector Index: {index_name}")
print(f"KB Role: {kb_role_name}")
print(f"KB Name: {kb_name}")

In [ ]:
# Step 1: Create S3 bucket
print("Creating S3 translation memory bucket...")
s3 = session.client('s3')
try:
    s3.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={'LocationConstraint': region}
    )
    print(f"✅ Created: {bucket_name}")
except ClientError as e:
    if 'BucketAlreadyOwnedByYou' in str(e):
        print(f"✅ Exists: {bucket_name}")
    else:
        raise

In [ ]:
# Step 2: Upload TM files
print("Uploading translation memory files...")
import os

memory_files = [
    "translation_memory_aws.csv"
]

data_dir = "data"

for filename in memory_files:
    file_path = os.path.join(data_dir, filename)
    try:
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                s3.put_object(
                    Bucket=bucket_name, 
                    Key=filename, 
                    Body=f.read(),
                    ContentType='text/csv'
                )
            print(f"✅ Uploaded: {filename}")
        else:
            print(f"⚠️ File not found: {file_path}")
    except Exception as e:
        print(f"❌ Failed to upload {filename}: {str(e)}")


#### Create S3 Vector bucket and Create Vector Index

In [ ]:
# Step 1: Create S3 Vector bucket
print("Creating S3 Vector bucket...")
s3vectors = session.client('s3vectors')
try:
    s3vectors.create_vector_bucket(vectorBucketName=vector_bucket_name)
    print(f"✅ Created vector bucket: {vector_bucket_name}")
except ClientError as e:
    if 'already exists' in str(e).lower():
        print(f"✅ Vector bucket exists: {vector_bucket_name}")
    else:
        raise

In [ ]:
# Step 2: Create vector index
print("Creating vector index...")
try:
    s3vectors.create_index(
        vectorBucketName=vector_bucket_name,
        indexName=index_name,
        dataType="float32",
        dimension=1024,
        distanceMetric="cosine"
    )
    print(f"✅ Created index: {index_name}")
except ClientError as e:
    if 'already exists' in str(e).lower():
        print(f"✅ Index exists: {index_name}")
    else:
        raise

#### Create Knowledge Base with S3 Data Source

TODO: Check notebook here: https://github.com/aws-samples/amazon-bedrock-samples/blob/main/rag/knowledge-bases/features-examples/03-optimizing-accuracy-retrieved-results/advanced_chunking_options.ipynb

In [ ]:
# Step 1: Create the IAM role for the Knowledge Base to assume
print("Creating IAM role for the Knowledge Base to use...")
iam = session.client('iam')

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel"],
            "Resource": f"arn:aws:bedrock:{region}::foundation-model/{embedding_model}"
        },
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{bucket_name}", f"arn:aws:s3:::{bucket_name}/*"]
        },
        {
            "Effect": "Allow",
            "Action": ["s3vectors:*"],
            "Resource": f"arn:aws:s3vectors:{region}:{account_id}:bucket/{vector_bucket_name}/index/{index_name}"
        }
    ]
}

try:
    role = iam.create_role(
        RoleName=kb_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    iam.put_role_policy(
        RoleName=kb_role_name,
        PolicyName="S3VectorsPolicy",
        PolicyDocument=json.dumps(policy)
    )
    print(f"✅ Created role: {kb_role_name}")
except ClientError as e:
    if 'EntityAlreadyExists' in str(e):
        role = iam.get_role(RoleName=kb_role_name)
        print(f"✅ Role exists: {kb_role_name}")
    else:
        raise

In [ ]:
# Step 2: Create Knowledge Base
print("Creating Knowledge Base...")
bedrock_agent = session.client('bedrock-agent')

# Wait for role propagation
time.sleep(10)

kb_response = bedrock_agent.create_knowledge_base(
    name=kb_name,
    description="KB with S3 Vectors",
    roleArn=role['Role']['Arn'],
    knowledgeBaseConfiguration={
        'type': "VECTOR",
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModelArn': f'arn:aws:bedrock:{region}::foundation-model/{embedding_model}'
        }
    },
    storageConfiguration={
        's3VectorsConfiguration': {
            'indexArn': f'arn:aws:s3vectors:{region}:{account_id}:bucket/{vector_bucket_name}/index/{index_name}'
        },
        'type': 'S3_VECTORS'
    }
)

kb_id = kb_response["knowledgeBase"]["knowledgeBaseId"]
print(f"✅ Created KB: {kb_id}")

# Wait for KB to be ready
while True:
    status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
    if status == "ACTIVE":
        print("✅ KB is ready")
        break
    print(f"Status: {status} - waiting...")
    time.sleep(10)

In [ ]:
# Step 3: Create a data source for the Knowledge Base
print("Creating a data source for the Knowledge Base...")
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name=f'{kb_id}-s3',
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{bucket_name}"
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {
                "maxTokens": 100,
                "overlapPercentage": 20
            }
        }
    }
)

ds_id = ds_response["dataSource"]["dataSourceId"]
print(f"✅ Created data source: {ds_id}")

In [ ]:
# Step 4: Start ingestion of TM from S3 data source
print("Starting ingestion of translation memory from S3 data source...")
job_response = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=ds_id
)

job_id = job_response["ingestionJob"]["ingestionJobId"]
print(f"✅ Started job: {job_id}")

# Wait for completion
while True:
    job = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=job_id
    )["ingestionJob"]
    
    status = job['status']
    if status == 'COMPLETE':
        print("✅ Ingestion complete")
        break
    elif status in ['FAILED', 'STOPPED']:
        print(f"❌ Ingestion failed: {status}")
        break
    
    print(f"Status: {status} - waiting...")
    time.sleep(30)

In [ ]:
# Test the Knowledge Base
print("Testing Knowledge Base...")
bedrock_runtime = session.client('bedrock-agent-runtime')

query = "What should I do if I can't use Bedrock?"
response = bedrock_runtime.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={'text': query},
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 3
        }
    }
)

print(f"\nQuery: {query}")
print("Results:")
for i, result in enumerate(response['retrievalResults'], 1):
    print(f"\n{i}. Score: {result['score']:.4f}")
    print(f"   Content: {result['content']['text']}")
    print(f"   Source: {result['location']['s3Location']['uri']}")

#### Deploy Chunking Lambda Function

Deploy the Lambda function using CloudFormation:

In [ ]:
# Deploy the Chunking Lambda function
!aws cloudformation deploy \
  --template-file deployment/chunking-lambda.yaml \
  --stack-name chunking-lambda-stack \
  --profile {profile_name} \
  --parameter-overrides \
    InputBucketName={bucket_name} \
    OutputBucketName={bucket_name} \
  --capabilities CAPABILITY_IAM

Python(97027) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
607175.82s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - chunking-lambda-stack


In [33]:
# Get the Lambda function ARN
cfn = session.client('cloudformation')
try:
    stack_outputs = cfn.describe_stacks(StackName='chunking-lambda-stack')['Stacks'][0]['Outputs']
    lambda_arn = next(o['OutputValue'] for o in stack_outputs if o['OutputKey'] == 'ChunkingLambdaFunctionArn')
    print(f"Lambda Function ARN: {lambda_arn}")
except Exception as e:
    print(f"Stack not found or not yet deployed: {e}")
%store lambda_arn

Lambda Function ARN: arn:aws:lambda:us-east-1:986528949439:function:ChunkingLambdaFunction
Stored 'lambda_arn' (str)


In [ ]:
print("Creating a data source for the Knowledge Base...")
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name=f'{kb_id}-s3-custom-chunking',
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{bucket_name}"
        }
    },
    vectorIngestionConfiguration={
        "customTransformationConfiguration": { 
            "intermediateStorage": { 
                "s3Location": { 
                "uri": "string"
                }
            },
            "transformations": [
                {
                    "transformationFunction": {
                        "lambdaConfiguration": {
                            "lambdaArn": {lambda_arn}
                        }
                    },
                    "stepToApply": "POST_CHUNKING"
                }
            ]
        },
        "chunkingConfiguration": {
            "chunkingStrategy": "NONE"
        }
    }
)

ds_id = ds_response["dataSource"]["dataSourceId"]
print(f"✅ Created data source: {ds_id}")

#### Ingest data source with custom chunking

### Custom Terminology

### Testing it out!

## Clean Up

Run the cleanup script to delete all resources:

In [ ]:
# Run cleanup script
!bash deployment/cleanup.sh